In [1]:
# Menyiapkan Dataset


import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


In [2]:
# A. EXTRACT 

from pyspark.sql import SparkSession

# 1. Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas7_2505060026_Pangestu Ilham Dwi Prastyo") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# 2. Proses Extract langsung dari HDFS
df_trx = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv", header=True, inferSchema=True)
df_cabang = spark.read.json("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json")
df_menu = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv", header=True, inferSchema=True)

# 3. Verifikasi jumlah baris
print("--- Proses Extract Selesai ---")
print("Jumlah transaksi :", df_trx.count())
print("Jumlah cabang    :", df_cabang.count())
print("Jumlah menu      :", df_menu.count())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/20 11:41:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/20 11:41:41 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
                                                                                

--- Proses Extract Selesai ---
Jumlah transaksi : 4000
Jumlah cabang    : 3
Jumlah menu      : 15


In [3]:
# B. TRANSFORM — Join & Kolom Turunan

from pyspark.sql.functions import col

# 1. Penggabungan Transaksi, Cabang, dan Menu (Inner Join)
df_gabungan = df_trx.join(df_cabang, on="cabang_id", how="inner") \
                    .join(df_menu, on="menu_id", how="inner")

# 2. Membuat kolom kalkulasi total_penjualan
df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))

df_gabungan.select("trx_id", "nama_cabang", "nama_menu", "qty", "harga", "total_penjualan").show(5)

+------+--------------------+---------+---+-----+---------------+
|trx_id|         nama_cabang|nama_menu|qty|harga|total_penjualan|
+------+--------------------+---------+---+-----+---------------+
|   KN0|Kopi Nusantara Ma...|  Menu-14|  4|18000|          72000|
|   KN1|Kopi Nusantara Ma...|  Menu-11|  4|22000|          88000|
|   KN2|Kopi Nusantara Ma...|  Menu-11|  3|22000|          66000|
|   KN3|Kopi Nusantara Se...|  Menu-11|  1|22000|          22000|
|   KN4|Kopi Nusantara Yo...|  Menu-12|  4|35000|         140000|
+------+--------------------+---------+---+-----+---------------+
only showing top 5 rows



In [4]:
# C. TRANSFORM — Window Function 

from pyspark.sql.functions import sum as spark_sum, row_number
from pyspark.sql.window import Window

# 1. Meringkas total penjualan per kombinasi cabang dan menu
ringkasan_menu_cabang = df_gabungan.groupBy("nama_cabang", "nama_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan_menu")
)

# 2. Menerapkan window function untuk mengambil top-2 menu pada setiap cabang
window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan_menu").desc())
top2_menu = ringkasan_menu_cabang.withColumn("peringkat", row_number().over(window_cabang)) \
                                 .filter(col("peringkat") <= 2)

top2_menu.orderBy("nama_cabang", "peringkat").show()

[Stage 19:>                                                         (0 + 1) / 1]

+--------------------+---------+--------------------+---------+
|         nama_cabang|nama_menu|total_penjualan_menu|peringkat|
+--------------------+---------+--------------------+---------+
|Kopi Nusantara Ma...|   Menu-1|             8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|             8085000|        2|
|Kopi Nusantara Se...|  Menu-13|             8750000|        1|
|Kopi Nusantara Se...|   Menu-9|             8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|             9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|             8160000|        2|
+--------------------+---------+--------------------+---------+



In [5]:
# D. TRANSFORM — Spark SQL 

# 1. Mendaftarkan DataFrame gabungan sebagai Temporary View
df_gabungan.createOrReplaceTempView("data_kopi")

# 2. Mengeksekusi query Spark SQL murni
kontribusi_barista = spark.sql('''
    SELECT kepala_barista, SUM(total_penjualan) AS kontribusi_penjualan
    FROM data_kopi
    GROUP BY kepala_barista
    ORDER BY kontribusi_penjualan DESC
''')

kontribusi_barista.show()

+--------------+--------------------+
|kepala_barista|kontribusi_penjualan|
+--------------+--------------------+
|         Nadia|           177680000|
|          Reza|            84645000|
+--------------+--------------------+



In [6]:
# E. LOAD

# 1. Memastikan direktori target tersedia (opsional)
!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/processed

# 2. Menyimpan data ke HDFS dalam format Parquet dan dipartisi berdasarkan kategori
df_gabungan.write.mode("overwrite").partitionBy("kategori_menu").parquet("hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan")

# 3. Membaca kembali dari HDFS untuk memverifikasi data
df_laporan = spark.read.parquet("hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan")
print("Jumlah baris data hasil akhir di HDFS:", df_laporan.count())

Jumlah baris data hasil akhir di HDFS: 4000


In [7]:
# F. Ringkasan Eksekutif & Rekomendasi 

from pyspark.sql.functions import count, avg

# Membuat tabel ringkasan eksekutif
ringkasan_eksekutif = df_laporan.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan_triwulan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan_triwulan").desc())

print("--- Tabel Ringkasan Eksekutif ---")
ringkasan_eksekutif.show(truncate=False)

--- Tabel Ringkasan Eksekutif ---
+-------------------------+----------+--------------+------------------------+----------------+-------------------------+
|nama_cabang              |kota      |kepala_barista|total_penjualan_triwulan|jumlah_transaksi|rata_rata_nilai_transaksi|
+-------------------------+----------+--------------+------------------------+----------------+-------------------------+
|Kopi Nusantara Semarang  |Semarang  |Nadia         |90542000                |1393            |64997.8463747308         |
|Kopi Nusantara Magelang  |Magelang  |Nadia         |87138000                |1302            |66926.26728110599        |
|Kopi Nusantara Yogyakarta|Yogyakarta|Reza          |84645000                |1305            |64862.06896551724        |
+-------------------------+----------+--------------+------------------------+----------------+-------------------------+

